In [1]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))

In [2]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score
from src.data_load_ionosphere import load_ionosphere
from src.preprocess import preprocess


In [3]:
seed = 42
qubits = 4

X_train, X_test, y_train, y_test = load_ionosphere(seed)
print("Shapes:", X_train.shape, X_test.shape)
print("Share of 'bad' in train:", (y_train == 0).mean())
print("Majority-class baseline accuracy:", max((y_test == 0).mean(), (y_test == 1).mean()))


Shapes: (280, 34) (71, 34)
Share of 'bad' in train: 0.3607142857142857
Majority-class baseline accuracy: 0.647887323943662


In [4]:
X_train_scale, X_test_scale, X_train_pca, X_test_pca, pca = preprocess(X_train, X_test, qubits)
print("PCA variance retained:", pca.explained_variance_ratio_.sum())


PCA variance retained: 0.5618398552710043


In [5]:
mm = MinMaxScaler(feature_range=(0, np.pi))
X_train_pca_scaled = mm.fit_transform(X_train_pca)
X_test_pca_scaled = np.clip(mm.transform(X_test_pca), 0, np.pi)


In [6]:
grid = {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.001, 0.01, 0.1, 1, 10, "scale"]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)


In [7]:
def tuned(Xtrain, Xtest, label):
    s = GridSearchCV(SVC(kernel="rbf"), grid, cv=cv, scoring="f1_macro").fit(Xtrain, y_train)
    p = s.best_estimator_.predict(Xtest)
    print(f"{label}: best={s.best_params_}  CV F1={s.best_score_:.3f}  "
          f"test acc={accuracy_score(y_test, p):.3f}  "
          f"recall(bad)={recall_score(y_test, p, pos_label=0):.3f}  "
          f"F1={f1_score(y_test, p, average='macro'):.3f}")



In [8]:
tuned(X_train_pca_scaled, X_test_pca_scaled, "Tuned RBF, PCA 4 (0-pi)")
tuned(X_train_scale, X_test_scale, "Tuned RBF, all 34 features")

Tuned RBF, PCA 4 (0-pi): best={'C': 100, 'gamma': 0.1}  CV F1=0.859  test acc=0.930  recall(bad)=0.840  F1=0.920
Tuned RBF, all 34 features: best={'C': 10, 'gamma': 0.1}  CV F1=0.946  test acc=0.930  recall(bad)=0.960  F1=0.925
